In [ ]:
from pathlib import Path
from fastkaggle import iskaggle, setup_comp

# Set these only when you want to send your results to your Kaggle account.
submit_to_kaggle = False
publish_to_kaggle = False

## Going faster

In [ ]:
comp = 'paddy-disease-classification'
path = Path(comp)
if iskaggle or not (path/'train_images').is_dir():
    path = setup_comp(comp)
from fastai.vision.all import *
set_seed(42)

In [ ]:
trn_path = Path('sml')

In [ ]:
resize_images(path/'train_images', dest=trn_path, max_size=256, recurse=True)

In [ ]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize((256,192)))

dls.show_batch(max_n=3)

In [ ]:
def train(arch, item, batch, epochs=5):
    dls = ImageDataLoaders.from_folder(trn_path, seed=42, valid_pct=0.2, item_tfms=item, batch_tfms=batch)
    learn = vision_learner(dls, arch, metrics=error_rate)
    # CUDA uses the lecture precision. The CPU kernel uses float32.
    if torch.cuda.is_available(): learn = learn.to_fp16()
    learn.fine_tune(epochs, 0.01)
    return learn

In [ ]:
learn = train('resnet26d', item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

## A ConvNeXt model

In [ ]:
arch = 'convnext_small.fb_in22k'

In [ ]:
learn = train(arch, item=Resize(192, method='squish'),
              batch=aug_transforms(size=128, min_scale=0.75))

## Preprocessing experiments

In [ ]:
learn = train(arch, item=Resize(192),
              batch=aug_transforms(size=128, min_scale=0.75))

In [ ]:
dls = ImageDataLoaders.from_folder(trn_path, valid_pct=0.2, seed=42,
    item_tfms=Resize(192, method=ResizeMethod.Pad, pad_mode=PadMode.Zeros))
dls.show_batch(max_n=3)

In [ ]:
learn = train(arch, item=Resize((256,192), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
      batch=aug_transforms(size=(171,128), min_scale=0.75))

## Test time augmentation

In [ ]:
valid = learn.dls.valid
preds,targs = learn.get_preds(dl=valid)

In [ ]:
error_rate(preds, targs)

In [ ]:
learn.dls.train.show_batch(max_n=6, unique=True)

In [ ]:
tta_preds,_ = learn.tta(dl=valid)

In [ ]:
error_rate(tta_preds, targs)

## Scaling up

In [ ]:
trn_path = path/'train_images'

In [ ]:
learn = train(arch, epochs=12,
              item=Resize((480, 360), method=ResizeMethod.Pad, pad_mode=PadMode.Zeros),
              batch=aug_transforms(size=(256,192), min_scale=0.75))

In [ ]:
tta_preds,targs = learn.tta(dl=learn.dls.valid)
error_rate(tta_preds, targs)

## Submission

In [ ]:
tst_files = get_image_files(path/'test_images').sorted()
tst_dl = learn.dls.test_dl(tst_files)

In [ ]:
preds,_ = learn.tta(dl=tst_dl)

In [ ]:
idxs = preds.argmax(dim=1)

In [ ]:
vocab = np.array(learn.dls.vocab)
results = pd.Series(vocab[idxs], name="idxs")

In [ ]:
ss = pd.read_csv(path/'sample_submission.csv')
ss['label'] = results
ss.to_csv('subm.csv', index=False)
print('\n'.join(Path('subm.csv').read_text().splitlines()[:10]))

In [ ]:
if submit_to_kaggle and not iskaggle:
    from kaggle import api
    api.competition_submit('subm.csv', 'convnext small 256x192 12 epochs tta', comp)

In [ ]:
# Optional: run from the directory containing this notebook.
if publish_to_kaggle and not iskaggle:
    import json, shutil
    from tempfile import TemporaryDirectory
    from fastkaggle import nb_meta
    from kaggle import api

    filename = '09-small-models-road-to-the-top-part-2.ipynb'
    meta = nb_meta(api.config_values['username'], 'small-models-road-to-the-top-part-2', 'Small models: Road to the Top, Part 2',
                   filename, competition=comp, private=True, gpu=True)
    with TemporaryDirectory() as folder:
        shutil.copyfile(filename, Path(folder)/filename)
        (Path(folder)/'kernel-metadata.json').write_text(json.dumps(meta), encoding='utf-8')
        api.kernels_push(folder)

## Conclusion